# Variantes de BRAM-EV — un mécanisme interne à la fois

Ce notebook **n'exécute aucune simulation** : il lit les artefacts d'une
campagne déjà produite.

```bash
python main.py run --config experiments/ablation_variants.yaml
```

Là où `ablation.ipynb` demande *« qu'apporte l'ajout de ce composant ? »*, ce
notebook demande *« ce mécanisme doit-il fonctionner comme il fonctionne ? »*.
Chaque variante remplace **un seul** mécanisme interne de la méthode complète
et se compare à `bramev` :

| Variante | Mécanisme neutralisé | Remplacé par |
| --- | --- | --- |
| `bramev_nearest_offer` | utilité multicritère | le véhicule prend l'offre la plus proche |
| `bramev_fixed_alpha` | hétérogénéité des alpha | un alpha commun (`--alpha-fixed`) |
| `bramev_global_rep` | réputation par société | un score unique partagé |
| `bramev_event_score` | score proportionnel à la durée | une pénalité forfaitaire par événement |

Le sens de lecture est **BRAM-EV -> variante** : un écart défavorable signifie
que le mécanisme neutralisé était utile. Le plan est déclaré une seule fois,
dans `src/experiments/methods.py`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

import pandas as pd
from IPython.display import Image, display

import src.experiments.methods as methods
from src.pipeline import ablation, figures
from src.pipeline.params import CaseParams
from src.pipeline.store import RunStore

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

## Choix du run

Il faut `bramev` **et** au moins une variante : sans la référence, aucun écart
n'est calculable. `latest_with_methods` prend le run le plus récent qui les
contient, et dit ce que contiennent les autres s'il n'en trouve aucun.

In [ ]:
for path in RunStore.list_runs('../results_grid'):
    present = sorted({row['method'] for row in RunStore(path).read_summary()})
    print(f"{path.name}\n    {', '.join(present) or 'aucun cas'}")

In [ ]:
store = RunStore.latest_with_methods(('bramev',) + methods.VARIANTS,
                                     '../results_grid')
params = store.read_params()
manifest = store.read_manifest()

print(store.root)
print(params.describe())
print(f"graine={params.seed} | commit={manifest['git_commit']} | "
      f"cas={manifest['nb_cases_done']}/{manifest['nb_cases_planned']}")
print(f"alpha imposé aux variantes à alpha fixe : {params.alpha_fixed}")

## Ce qui est réellement neutralisé

Contrôle préalable : chaque variante doit différer de `bramev` par **exactement
un** mécanisme. Ces colonnes viennent des drapeaux effectivement appliqués
pendant la simulation (`src/pipeline/tables.py`), pas de l'intention déclarée.

In [ ]:
summary = pd.read_csv(store.summary_path)

ORDRE = ['bramev'] + list(methods.VARIANTS)
summary['method'] = pd.Categorical(summary['method'], ORDRE + [
    m for m in summary['method'].unique() if m not in ORDRE], ordered=True)

variantes = summary[summary['method'].isin(ORDRE)].copy()

plan = (variantes[['method', 'method_label', 'method_family', 'offer_choice',
                   'alpha_mode', 'reputation_scope', 'score_weighting',
                   'broadcast', 'reputation', 'adaptation']]
        .drop_duplicates()
        .sort_values('method')
        .set_index('method'))
plan

In [ ]:
champs = ['broadcast', 'reputation', 'adaptation', 'offer_choice',
          'alpha_mode', 'reputation_scope', 'score_weighting']
reference = plan.loc['bramev']

for nom in methods.VARIANTS:
    if nom not in plan.index:
        print(f"ABSENT    {nom}")
        continue
    change = [c for c in champs if plan.loc[nom, c] != reference[c]]
    etat = 'OK' if len(change) == 1 else 'ANOMALIE'
    print(f"{etat:9} {nom:22} change={', '.join(change):<20} "
          f"mécanisme={ablation.VARIANT_MECHANISM.get(nom, '?')}")

In [ ]:
# Décomposition calculée à la volée depuis summary.csv. Le pipeline persiste
# exactement les mêmes tables (`ablation.csv`, `ablation_mean.csv`) et les
# réécrit à chaque cas ; les recalculer ici rend le notebook utilisable sur une
# campagne encore en cours, interrompue, ou antérieure à l'étude d'ablation.
detail = pd.DataFrame(ablation.detail_rows(summary.to_dict('records')))
moyennes = pd.DataFrame(ablation.mean_rows(detail.to_dict('records')))

print(f"{len(detail)} écarts calculés sur "
      f"{detail[['scenario', 'nb_cars']].drop_duplicates().shape[0]} mondes")

## La méthode complète face à chaque variante

Une ligne par méthode, moyenne sur tous les mondes du run. `bramev` est la
référence : lire les autres lignes comme des écarts à celle-ci.

In [ ]:
METRIQUES = ['exact_satisfaction', 'rate_abs', 'mean_service_rate',
             'slot_waste_rate', 'nb_reservations', 'mean_waiting_time_min',
             'mean_travel_distance_km', 'mean_offers_per_demand',
             'total_ms_mean']

niveaux = (variantes.groupby('method', observed=True)[METRIQUES]
                    .mean()
                    .rename(index=methods.label)
                    .round(4))
niveaux

In [ ]:
# Écart relatif à bramev, en pourcentage. Le signe est brut : la direction
# propre à chaque métrique est appliquée plus bas par `improvement`.
reference_valeurs = variantes[variantes['method'] == 'bramev'][METRIQUES].mean()
ecarts = (variantes.groupby('method', observed=True)[METRIQUES].mean()
          .div(reference_valeurs) - 1.) * 100.
ecarts.drop(index='bramev').rename(index=methods.label).round(2)

## Effet du mécanisme neutralisé

`ablation_mean.csv`, filtré sur `kind == 'variant'`. `share_improved` est la
part des mondes où **neutraliser** le mécanisme améliore la métrique : une
valeur basse est donc un argument *pour* le mécanisme.

In [ ]:
print(ablation.render_mean_table(moyennes.to_dict('records')))

In [ ]:
bloc = moyennes[moyennes['kind'] == 'variant']

effets = bloc.pivot_table(index=['component', 'to_method'],
                          columns='metric_label',
                          values=['mean_delta_pct', 'share_improved'])
effets.round(3)

In [ ]:
# Verdict par mécanisme : sur combien de (monde x métrique) le neutraliser
# dégrade-t-il le résultat ? Une part élevée plaide pour le mécanisme.
bloc_detail = detail[detail['kind'] == 'variant']

verdict = (bloc_detail.groupby(['component', 'to_method'])
           .agg(comparaisons=('improvement', 'size'),
                neutraliser_dégrade=('improvement', lambda s: (~s).mean()))
           .round(3)
           .sort_values('neutraliser_dégrade', ascending=False))
verdict

## Dispersion par variante

Comme pour l'échelle, une moyenne peut être portée par un seul monde.

In [ ]:
for metrique in ['exact_satisfaction', 'rate_abs', 'mean_service_rate']:
    sous = bloc_detail[bloc_detail['metric'] == metrique]
    if sous.empty:
        continue
    print(f"\n=== {sous['metric_label'].iloc[0]} — écart à bramev ===")
    stats = (sous.groupby('component')['delta']
                 .agg(['count', 'min', 'median', 'mean', 'max'])
                 .round(6))
    stats['mondes_améliorés'] = sous.groupby('component')['improvement'].mean().round(3)
    display(stats)

## Vérifications propres à chaque variante

Une variante peut afficher un écart nul simplement parce que son mécanisme
n'était pas sollicité. Les quatre contrôles ci-dessous distinguent
« mécanisme inutile » de « mécanisme jamais testé ».

### `bramev_nearest_offer` — le classement avait-il de quoi trancher ?

Choisir par utilité plutôt que par distance ne change rien si chaque demande ne
reçoit qu'une offre. `mean_offers_per_demand` dit si la comparaison a une
substance.

In [ ]:
offres = (variantes.groupby('method', observed=True)['mean_offers_per_demand']
                   .mean().round(3))
print(offres, end='\n\n')

if offres.get('bramev', 0.) <= 1.:
    print("ATTENTION : au plus une offre par demande en moyenne — le critère "
          "de sélection n'a presque jamais eu à choisir. Un écart nul ne dit "
          "rien de l'utilité multicritère.")
else:
    print(f"{offres['bramev']:.2f} offre(s) par demande : le classement a "
          "effectivement eu à trancher.")

### `bramev_fixed_alpha` — l'hétérogénéité a-t-elle été supprimée ?

La table `alpha` porte la trajectoire de l'arbitrage profit/risque par station.
Sous `bramev` les alpha sont dispersés et convergent par apprentissage
collectif ; sous `bramev_fixed_alpha` ils doivent tous valoir `alpha_fixed`,
ce qui rend l'apprentissage inerte — c'est voulu, et c'est ce qui isole la
contribution de l'hétérogénéité elle-même.

In [ ]:
scenario, nb_cars = params.scenarios[-1], params.fleet_sizes[-1]

def lire_alpha(methode):
    chemin = store.table_path(CaseParams(scenario, nb_cars, methode), 'alpha')
    return pd.read_csv(chemin) if chemin.is_file() else None

for methode in ('bramev', 'bramev_fixed_alpha'):
    table = lire_alpha(methode)
    if table is None:
        print(f"{methode:20} (table absente)")
        continue
    initial = table[table['update_step'] == 0]['alpha']
    final = table[table['update_step'] == table['update_step'].max()]['alpha']
    print(f"{methode:20} étapes={table['update_step'].max() + 1:2}  "
          f"alpha initial: {initial.min():.3f}–{initial.max():.3f} "
          f"(écart-type {initial.std():.4f})  ->  "
          f"final: {final.min():.3f}–{final.max():.3f} "
          f"(écart-type {final.std():.4f})")

In [ ]:
# Trajectoire moyenne et dispersion, étape d'apprentissage par étape.
trajectoires = {}
for methode in ('bramev', 'bramev_fixed_alpha'):
    table = lire_alpha(methode)
    if table is None:
        continue
    trajectoires[methode] = table.groupby('update_step')['alpha'].agg(
        moyenne='mean', ecart_type='std', minimum='min', maximum='max')

if trajectoires:
    display(pd.concat(trajectoires, axis=1).round(4))

### `bramev_global_rep` — le score est-il devenu un bien public ?

Sous `bramev`, un véhicule porte un score par société (`score_index` = la
société propriétaire) : une mauvaise réputation chez l'un ne se transmet pas à
l'autre. Sous `bramev_global_rep`, toutes les stations écrivent et lisent la
même case, donc `score_index` vaut 0 partout.

In [ ]:
for methode in ('bramev', 'bramev_global_rep'):
    chemin = store.table_path(CaseParams(scenario, nb_cars, methode), 'stations')
    if not chemin.is_file():
        print(f"{methode:20} (table absente)")
        continue
    stations = pd.read_csv(chemin)
    print(f"{methode:20} score_index={sorted(stations['score_index'].unique())}  "
          f"pondération={sorted(stations['score_weighting'].unique())}  "
          f"sociétés={sorted(stations['society_id'].unique())}")

In [ ]:
# Conséquence attendue : un score partagé se dégrade plus vite pour un même
# véhicule, donc les stations rejettent davantage et servent moins.
(variantes[variantes['method'].isin(['bramev', 'bramev_global_rep'])]
 .groupby('method', observed=True)
 .agg(demandes_rejetees=('nb_station_level_rejections', 'mean'),
      offres_par_demande=('mean_offers_per_demand', 'mean'),
      reservations=('nb_reservations', 'mean'),
      taux_de_service=('mean_service_rate', 'mean'))
 .round(3))

### `bramev_event_score` — la durée pesait-elle vraiment ?

Une pénalité forfaitaire n'est différente d'une pénalité proportionnelle que si
les durées réservées varient. `d_prop` moyen et sa dispersion, lus dans la
table des acceptations, disent si l'écart entre les deux pondérations pouvait
se manifester.

In [ ]:
chemin = store.table_path(CaseParams(scenario, nb_cars, 'bramev'), 'acceptances')
if chemin.is_file():
    acceptations = pd.read_csv(chemin)
    print(f"{len(acceptations)} offres acceptées")
    display(acceptations[['distance_km', 'waiting_time_min']].describe().round(3))
else:
    print('table acceptances absente (--no-save-tables ?)')

# Le rapport des pénalités vaut exactement d_n entre les deux pondérations
# (cf. Station.update_car_score) : plus les durées sont dispersées, plus les
# deux régimes divergent.
config = next(store.iter_results())['config']
print("\npoints de stratégie (société) :", config['base_points_strategy'])

## Figures

In [ ]:
chemin = store.figure_path('ablation_variants')
if chemin.is_file():
    print(chemin.name)
    display(Image(filename=str(chemin)))

In [ ]:
figures.fig_ablation_variants(summary.to_dict('records'))

## Santé du run

In [ ]:
print('invariants OK :', bool(summary['invariant_ok'].all()))
print('réservations non résolues :', int(summary['nb_unresolved'].sum()))
print('pannes :', int(summary['nb_breakdowns'].sum()))

vus = set()
for resultat in store.iter_results():
    for message in resultat['behaviors'].get('diagnostics', []):
        if message not in vus:
            vus.add(message)
            print(f"\n[diagnostic] {message}")